# WordNet Description Length Analysis

Notebook nay phan tich phan phoi so luong tu trong moi text description cua entity,
va tinh do dai trung binh tren tap `wordnet-mlj12-definitions.txt`.

In [ ]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt

# Optional style for clearer plots
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
data_path = Path('../data/wn18rr/wordnet-mlj12-definitions.txt')
if not data_path.exists():
    raise FileNotFoundError(f'Cannot find file: {data_path.resolve()}')

# The file format is expected as: synset_id<TAB>lemma<TAB>definition
df = pd.read_csv(
    data_path,
    sep='\t',
    header=None,
    names=['synset_id', 'lemma', 'definition'],
    dtype=str,
    keep_default_na=False
)

# Use definition as the description text; if missing, fallback to lemma
df['description_text'] = df['definition'].where(df['definition'].str.strip().ne(''), df['lemma'])

# Count words using a regex that captures alphanumeric word tokens
word_pattern = re.compile(r"\b[\w']+\b")
df['word_count'] = df['description_text'].apply(lambda s: len(word_pattern.findall(str(s))))

df[['synset_id', 'lemma', 'description_text', 'word_count']].head()

In [ ]:
summary = {
    'num_entities': int(len(df)),
    'avg_word_count': float(df['word_count'].mean()),
    'median_word_count': float(df['word_count'].median()),
    'min_word_count': int(df['word_count'].min()),
    'max_word_count': int(df['word_count'].max()),
    'std_word_count': float(df['word_count'].std())
}

print('Summary statistics for description length (in words):')
for k, v in summary.items():
    if isinstance(v, float):
        print(f'- {k}: {v:.2f}')
    else:
        print(f'- {k}: {v}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['word_count'], bins=50, edgecolor='black', alpha=0.85)
ax.axvline(df['word_count'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean = {df['word_count'].mean():.2f}")
ax.set_title('Distribution of Word Counts per Entity Description')
ax.set_xlabel('Word count in description')
ax.set_ylabel('Number of entities')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Optional: inspect extreme examples
print('Top 5 longest descriptions by word count:')
display(df.nlargest(5, 'word_count')[['synset_id', 'lemma', 'word_count', 'description_text']])

print('Top 5 shortest descriptions by word count:')
display(df.nsmallest(5, 'word_count')[['synset_id', 'lemma', 'word_count', 'description_text']])